 ## Breast Cancer Risk Prediction on Encrypted Data
 ### **Eight encrypted ML models + clinical benchmarks**

 **Scenario:**
  Population health research requires combining breast cancer screening data across hospitals
  and health systems. Researchers want to compare risk-based screening (identifying high-risk
  women for earlier/more frequent mammography) against fixed-schedule screening.

 **Challenge:**
  Patient records contain protected health information (PHI) -- age, family history, biopsy
  results, breast density -- that cannot be shared across institutions under HIPAA.

 **Solution:**
  Train eight privacy-preserving ML models from encrypted aggregate counts only, then keep the
  clinical **BCRAT (Gail Model)** and **BCSC** risk calculator comparison separate.

 **Key insight (Paige et al. 2023):**
  Published risk models disagree on ~20% of individual women. A privacy-preserving model
  trained on multi-institutional encrypted data can participate in these comparisons
  *without any institution sharing raw PHI*.


 ### Import Dependencies

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os, time
from IPython.display import display, HTML
from blind_ml.client import BlindInsightClient
from blind_ml.healthcare import (
    load_env, get_bc_demo_config,
    load_bc_training_data, load_bc_test_data,
    discover_feature_values, get_bc_feature_values,
    run_bc_conditional_queries, build_bc_model,
    train_plaintext_bc_nb, bc_training_summary_table,
    compute_bc_metrics, bc_model_summary_table, bc_confusion_matrix_html,
    bc_eight_model_table,
    run_model_comparison, sample_comparison_table,
    run_bc_realtime_demo,
)

# --- Configuration (static defaults + env overrides) ---
cfg = get_bc_demo_config()
load_env()
PROXY_URL = os.environ.get("BI_PROXY_URL", "https://local.blindinsight.io")
ORG = os.environ.get("BI_ORG", "your-org-slug")

DATASET = cfg["dataset"]
SCHEMA = cfg["schema"]
TEST_SCHEMA = cfg["test_schema"]
SQLITE_DB = cfg["sqlite_db"]
TEST_SQLITE_DB = cfg["test_sqlite_db"]
NB_FEATURES = cfg["nb_features"]
TARGET = cfg["target"]
CLINICAL_THRESHOLD = cfg["clinical_threshold"]
POP_5YR_RATE = cfg["pop_5yr_rate"]
HIPAA_K = 11

# --- Connect to Blind Insight proxy ---
client = BlindInsightClient(proxy_url=PROXY_URL, verify_ssl=False)
client.warm_up(ORG, DATASET, SCHEMA, preflight_filters=[
    ["cancer_5yr:1"],
    ["cancer_5yr:0"],
    ["cancer_5yr:1", "age_group:50_59"],
    ["cancer_5yr:1", "num_first_degree_relatives:1"],
    ["cancer_5yr:1", "num_prior_biopsies:0"],
    ["cancer_5yr:1", "age_at_first_birth:20~24"],
    ["cancer_5yr:1", "breast_density:3"],
])

# --- Load plaintext mirrors for local benchmarking ---
df, n_total = load_bc_training_data(SQLITE_DB)
df_test_local, _ = load_bc_test_data(TEST_SQLITE_DB)
print(f"Loaded {n_total:,} training records, {len(df_test_local):,} test records")


  Proxy warm-up: schema (3770ms) + 7 index preflights (20031ms) = 23801ms total
Loaded 100,000 training records, 10,000 test records


 ### Training Set Sample Records

In [2]:
from blind_ml.demo_helpers import data_table
display(HTML(data_table(
    df,
    columns=["age", "age_group", "num_first_degree_relatives", "breast_density",
             "menarche_category", "age_at_first_birth", "cancer_5yr"],
    limit=5,
    caption="Breast Cancer Screening Data Sample",
    number_cols=["age", "breast_density", "num_first_degree_relatives", "age_at_first_birth", "cancer_5yr"],
    footer="Target: cancer_5yr = 1 (developed invasive breast cancer within 5 years)",
)))

age,age_group,num_first_degree_relatives,breast_density,menarche_category,age_at_first_birth,cancer_5yr
44,40_49,0,2,under_12,31,0
46,40_49,1,4,12_13,23,0
45,40_49,1,2,under_12,17,0
48,40_49,0,1,12_13,25,0
53,50_59,1,3,under_12,26,0


 ### Train Naive Bayes on Encrypted Data

 We train the NB classifier using only **encrypted aggregate counts** from Blind Insight.
 No individual patient record is decrypted during training.

 A plaintext NB is trained on the local mirror for comparison -- they should produce
 identical probability tables.

In [3]:
from blind_ml.healthcare import (bc_naive_bayes_risk)

print("Training models...")
feature_values = get_bc_feature_values()

print("  Blind Insight Encrypted Naive Bayes...", end=" ", flush=True)
enc_train_start = time.time()
client.profiling.enable()
bi_raw = run_bc_conditional_queries(
    client, ORG, DATASET, SCHEMA, feature_values, include_base_rates=True,
    min_cell_size=HIPAA_K,
)
client.profiling.disable()
enc_train_time = time.time() - enc_train_start
enc_queries = bi_raw["enc_queries"] + 2  # conditional + 2 base-rate queries
n_cancer_bi = bi_raw["n_cancer"]
n_no_cancer_bi = bi_raw["n_no_cancer"]
n_total_bi = n_cancer_bi + n_no_cancer_bi
cohort_prev = n_cancer_bi / n_total_bi if n_total_bi else 0.0
print(f"done {enc_queries} encrypted aggregate queries ({enc_train_time:.1f}s)")
print(f"  HIPAA/CMS k={bi_raw['min_cell_size']} suppressed cells: {bi_raw['n_suppressed']}")

df_train = df.head(n_total_bi)

print("  Plaintext Naive Bayes...", end=" ", flush=True)
plain_start = time.time()
plain_nb = train_plaintext_bc_nb(df_train, feature_values)
plain_train_time = time.time() - plain_start
print(f"done ({plain_train_time * 1000:.0f}ms)")

# BI-sourced, k-suppressed aggregate marginals reused by later encrypted models.
raw_results = bi_raw["raw_results"]
bi = build_bc_model(raw_results, feature_values, n_cancer_bi, n_no_cancer_bi)
P_enc = bi["P"]
P_plain = plain_nb["P"]

# Keep encrypted NB likelihoods while calibrating to population-level incidence.
P_cancer = P_cancer_plain = POP_5YR_RATE
P_no_cancer = P_no_cancer_plain = 1.0 - POP_5YR_RATE

n_cancer_local = int(df_train["cancer_5yr"].sum())
n_no_cancer_local = len(df_train) - n_cancer_local

display(HTML(bc_training_summary_table(
    n_cancer_local, n_no_cancer_local,
    n_cancer_bi, n_no_cancer_bi,
    enc_queries, enc_train_time, plain_train_time,
)))
print()
print(f"{enc_queries} encrypted aggregate queries")
print(f"Suppression policy: {bi_raw['suppression_policy']}")
print(f"Cohort prevalence: {bi['P_cancer']:.3f}  ->  recalibrated to population rate: {POP_5YR_RATE}")

# Fraud-style model metrics on local held-out mirror; evaluation only, not encrypted training.
y_true_nb = [int(v) for v in df_test_local["is_cancer"].values]
nb_enc_scores, nb_plain_scores = [], []
for _, row in df_test_local.iterrows():
    r = row.to_dict()
    nb_enc_scores.append(bc_naive_bayes_risk(P_cancer, P_no_cancer, P_enc, r))
    nb_plain_scores.append(bc_naive_bayes_risk(P_cancer_plain, P_no_cancer_plain, P_plain, r))

nb_m = compute_bc_metrics(
    y_true_nb, nb_enc_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)
nb_plain_m = compute_bc_metrics(
    y_true_nb, nb_plain_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)

display(HTML(bc_model_summary_table(
    "Naive Bayes",
    enc_metrics=nb_m,
    plain_metrics=nb_plain_m,
    enc_train_time=enc_train_time,
    plain_train_time=plain_train_time,
    enc_queries=enc_queries,
    plain_label="Plaintext NB",
)))
display(HTML(bc_confusion_matrix_html("Naive Bayes", nb_m, nb_plain_m)))


Training models...
  Blind Insight Encrypted Naive Bayes... done 70 encrypted aggregate queries (133.4s)
  HIPAA/CMS k=11 suppressed cells: 0
  Plaintext Naive Bayes... done (603ms)


,Plaintext,Blind Insight,Overhead
Cancer (5yr),"3,543","3,543",-
No Cancer,"96,457","96,457",-
Total,"100,000","100,000",-
Queries,0,70,-
Train Time,0.603149s,133.4s,+132.8s
Data Decrypted,YES,NEVER,-



70 encrypted aggregate queries
Suppression policy: cms_k11_fixed_midpoint_5
Cohort prevalence: 0.035  ->  recalibrated to population rate: 0.016


,Plaintext NB,Blind Insight,Delta
F1 @1.67% risk,0.112,0.112,+0.000
F1@best,0.126,0.126,+0.000
ROC-AUC,0.675,0.675,+0.000
PR-AUC,0.074,0.074,+0.000
F1@best @ 1.6% pop prior,0.126,0.126,+0.000
Sensitivity,58.5%,58.5%,+0.0pp
Specificity,65.5%,65.5%,+0.0pp
PPV (precision),6.2%,6.2%,+0.0pp
Flagged High-Risk,35.4%,35.4%,+0.0pp
Accuracy,65.2%,65.2%,+0.0pp


,Pred Low,Pred High
Actual No,"6,303","3,321"
Actual Yes,156,220
,Pred Low,Pred High
Actual No,"6,303","3,321"
Actual Yes,156,220


### Train Gaussian Naive Bayes on Encrypted Aggregates

Gaussian Naive Bayes models numeric/ordinal clinical fields with class-specific means and variances. The encrypted path uses only BI value-count aggregates with HIPAA/CMS k=11 suppression before sufficient statistics are computed.


In [4]:
from blind_ml.healthcare import (
    run_encrypted_gnb_bc, bc_gnb_predict,
    train_plaintext_gnb_bc, bc_plaintext_gnb_predict_proba, recalibrate_risk
)

print("Training Gaussian Naive Bayes...")
GNB_FEATURES = [
    "age",
    "age_at_menarche",
    "age_at_first_birth",
    "num_first_degree_relatives",
    "num_prior_biopsies",
    "breast_density",
]

print("  Blind Insight Encrypted GaussianNB...", end=" ", flush=True)
gnb_enc = run_encrypted_gnb_bc(
    client, ORG, DATASET, SCHEMA,
    numeric_features=GNB_FEATURES,
    n_cancer=n_cancer_bi, n_no_cancer=n_no_cancer_bi,
    min_cell_size=HIPAA_K,
)
print(f"done ({gnb_enc['train_time']:.1f}s)")
print(f"  HIPAA/CMS k={gnb_enc['min_cell_size']} suppressed cells: {gnb_enc['n_suppressed']}")

print("  sklearn GaussianNB...", end=" ", flush=True)
gnb_plain = train_plaintext_gnb_bc(df_train, numeric_features=GNB_FEATURES)
print(f"done ({gnb_plain['train_time'] * 1000:.0f}ms)")

display(HTML(bc_training_summary_table(
    gnb_plain["n_cancer"], gnb_plain["n_no_cancer"],
    gnb_enc["n_cancer"], gnb_enc["n_no_cancer"],
    gnb_enc["enc_queries"], gnb_enc["train_time"], gnb_plain["train_time"],
)))

print()
print(f"{gnb_enc['enc_queries']} encrypted aggregate queries")
print(f"Suppression policy: {gnb_enc['suppression_policy']}")

# Evaluate on local held-out mirror; this is benchmark/evaluation only, not encrypted training.
y_true_gnb = [int(v) for v in df_test_local["is_cancer"].values]

gnb_enc_scores = []
for _, row in df_test_local.iterrows():
    _, risk = bc_gnb_predict(gnb_enc, row.to_dict())
    gnb_enc_scores.append(recalibrate_risk(risk, cohort_prev, POP_5YR_RATE))

gnb_plain_proba = bc_plaintext_gnb_predict_proba(
    gnb_plain["model"], gnb_plain["features"], df_test_local,
)
gnb_plain_scores = [recalibrate_risk(prob, cohort_prev, POP_5YR_RATE) for prob in gnb_plain_proba]

gnb_enc_m = compute_bc_metrics(
    y_true_gnb, gnb_enc_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)
gnb_plain_m = compute_bc_metrics(
    y_true_gnb, gnb_plain_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)

print()
print(
    f"Encrypted GaussianNB F1={gnb_enc_m['f1']:.3f} ROC-AUC={gnb_enc_m['roc_auc']:.3f} | "
    f"sklearn F1={gnb_plain_m['f1']:.3f} ROC-AUC={gnb_plain_m['roc_auc']:.3f}"
)
display(HTML(bc_model_summary_table(
    "Gaussian Naive Bayes",
    enc_metrics=gnb_enc_m,
    plain_metrics=gnb_plain_m,
    enc_train_time=gnb_enc["train_time"],
    plain_train_time=gnb_plain["train_time"],
    enc_queries=gnb_enc["enc_queries"],
    plain_label="sklearn GaussianNB",
)))
display(HTML(bc_confusion_matrix_html("Gaussian Naive Bayes", gnb_enc_m, gnb_plain_m)))


Training Gaussian Naive Bayes...
  Blind Insight Encrypted GaussianNB... done (107.4s)
  HIPAA/CMS k=11 suppressed cells: 2
  sklearn GaussianNB... done (13ms)


,Plaintext,Blind Insight,Overhead
Cancer (5yr),"3,543","3,543",-
No Cancer,"96,457","96,457",-
Total,"100,000","100,000",-
Queries,0,234,-
Train Time,0.012935s,107.4s,+107.4s
Data Decrypted,YES,NEVER,-



234 encrypted aggregate queries
Suppression policy: cms_k11_fixed_midpoint_5

Encrypted GaussianNB F1=0.103 ROC-AUC=0.648 | sklearn F1=0.116 ROC-AUC=0.670


,sklearn GaussianNB,Blind Insight,Delta
F1 @1.67% risk,0.116,0.103,-0.013
F1@best,0.123,0.129,+0.006
ROC-AUC,0.670,0.648,-0.022
PR-AUC,0.073,0.066,-0.007
F1@best @ 1.6% pop prior,0.123,0.129,+0.006
Sensitivity,43.6%,67.6%,+23.9pp
Specificity,76.2%,55.1%,-21.1pp
PPV (precision),6.7%,5.5%,-1.1pp
Flagged High-Risk,24.6%,45.8%,+21.2pp
Accuracy,74.9%,55.5%,-19.4pp


,Pred Low,Pred High
Actual No,"5,299","4,325"
Actual Yes,122,254
,Pred Low,Pred High
Actual No,"7,330","2,294"
Actual Yes,212,164


### Train Bayesian Network on Encrypted CPT Aggregates

Bayesian Network training uses class-split CPT aggregate counts from Blind Insight. Parent-conditioned cells are HIPAA/CMS k-suppressed before fitting; local data is used only for the plaintext benchmark and held-out evaluation.


In [5]:
from blind_ml.healthcare import (
    run_encrypted_bn_bc, bc_bn_predict,
    train_plaintext_bn_bc, bc_plaintext_bn_predict_proba,
)

print("Training Bayesian Network...")
print("  Blind Insight Encrypted BN...", end=" ", flush=True)
bn_enc = run_encrypted_bn_bc(
    client, ORG, DATASET, SCHEMA,
    feature_values=feature_values,
    n_cancer=n_cancer_bi, n_no_cancer=n_no_cancer_bi,
    min_cell_size=HIPAA_K,
)
print(f"done ({bn_enc['train_time']:.1f}s)")
print(f"  HIPAA/CMS k={bn_enc['min_cell_size']} suppressed cells: {bn_enc['n_suppressed']}")

print("  Plaintext BN benchmark...", end=" ", flush=True)
bn_plain = train_plaintext_bn_bc(df_train, feature_values)
print(f"done ({bn_plain['train_time'] * 1000:.0f}ms)")

display(HTML(bc_training_summary_table(
    bn_plain["n_cancer"], bn_plain["n_no_cancer"],
    bn_enc["n_cancer"], bn_enc["n_no_cancer"],
    bn_enc["enc_queries"], bn_enc["train_time"], bn_plain["train_time"],
)))

print()
print(f"{bn_enc['enc_queries']} encrypted aggregate queries")
print(f"Suppression policy: {bn_enc['suppression_policy']}")

# Evaluate on local held-out mirror; this is benchmark/evaluation only, not encrypted training.
y_true_bn = [int(v) for v in df_test_local["is_cancer"].values]

bn_enc_scores = []
for _, row in df_test_local.iterrows():
    _, risk = bc_bn_predict(bn_enc, row.to_dict())
    bn_enc_scores.append(recalibrate_risk(risk, cohort_prev, POP_5YR_RATE))

bn_plain_proba = bc_plaintext_bn_predict_proba(bn_plain, df_test_local)
bn_plain_scores = [recalibrate_risk(prob, cohort_prev, POP_5YR_RATE) for prob in bn_plain_proba]

bn_enc_m = compute_bc_metrics(
    y_true_bn, bn_enc_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)
bn_plain_m = compute_bc_metrics(
    y_true_bn, bn_plain_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)

print()
print(
    f"Encrypted BN F1={bn_enc_m['f1']:.3f} ROC-AUC={bn_enc_m['roc_auc']:.3f} | "
    f"plaintext BN F1={bn_plain_m['f1']:.3f} ROC-AUC={bn_plain_m['roc_auc']:.3f}"
)
display(HTML(bc_model_summary_table(
    "Bayesian Network",
    enc_metrics=bn_enc_m,
    plain_metrics=bn_plain_m,
    enc_train_time=bn_enc["train_time"],
    plain_train_time=bn_plain["train_time"],
    enc_queries=bn_enc["enc_queries"],
    plain_label="Plaintext BN",
)))
display(HTML(bc_confusion_matrix_html("Bayesian Network", bn_enc_m, bn_plain_m)))


Training Bayesian Network...
  Blind Insight Encrypted BN... done (84.1s)
  HIPAA/CMS k=11 suppressed cells: 1
  Plaintext BN benchmark... done (170ms)


,Plaintext,Blind Insight,Overhead
Cancer (5yr),"3,543","3,543",-
No Cancer,"96,457","96,457",-
Total,"100,000","100,000",-
Queries,0,140,-
Train Time,0.170255s,84.1s,+83.9s
Data Decrypted,YES,NEVER,-



140 encrypted aggregate queries
Suppression policy: cms_k11_fixed_midpoint_5

Encrypted BN F1=0.107 ROC-AUC=0.678 | plaintext BN F1=0.114 ROC-AUC=0.678


,Plaintext BN,Blind Insight,Delta
F1 @1.67% risk,0.114,0.107,-0.006
F1@best,0.136,0.136,+0.000
ROC-AUC,0.678,0.678,-0.000
PR-AUC,0.078,0.078,-0.000
F1@best @ 1.6% pop prior,0.136,0.136,+0.000
Sensitivity,61.2%,63.3%,+2.1pp
Specificity,64.3%,60.3%,-4.0pp
PPV (precision),6.3%,5.9%,-0.4pp
Flagged High-Risk,36.7%,40.6%,+3.9pp
Accuracy,64.2%,60.4%,-3.8pp


,Pred Low,Pred High
Actual No,"5,800","3,824"
Actual Yes,138,238
,Pred Low,Pred High
Actual No,"6,186","3,438"
Actual Yes,146,230


### Encrypted Decision Tree from BI Aggregate Counts

The encrypted Decision Tree trains with `DecisionTreeModel.fit_from_counts(...)` using BI aggregate counts only. NB marginals seed the root cache, deeper path counts are queried as aggregates, and `HIPAA_K` blocks unsafe small-positive splits.


In [6]:
from blind_ml.healthcare import (
    run_encrypted_dt_bc, bc_dt_predict, bc_dt_describe,
    train_plaintext_bc_dt, plaintext_predict_proba,
)

cohort_prev = n_cancer_bi / n_total_bi if n_total_bi else 0.0
raw_results = bi_raw["raw_results"]

print("Training Decision Trees...")
print("  Blind Insight Encrypted DT (aggregate counts only)...", end=" ", flush=True)
dt = run_encrypted_dt_bc(
    client, ORG, DATASET, SCHEMA,
    feature_values=feature_values,
    raw_results=raw_results,
    n_cancer=n_cancer_bi, n_no_cancer=n_no_cancer_bi,
    k_min=HIPAA_K,
)
print(f"done ({dt['train_time']:.2f}s)")
print(
    f"  Reused {len(raw_results)} NB marginals; "
    f"additional DT aggregate queries: {dt['additional_dt_queries']}"
)

print("  Plaintext DT (sklearn CART)...", end=" ", flush=True)
plain_dt = train_plaintext_bc_dt(df_train, feature_values, max_depth=3)
print(f"done ({plain_dt['train_time'] * 1000:.0f}ms)")
print("\n" + bc_dt_describe(dt))

# Evaluate on local held-out mirror; this is benchmark/evaluation only, not encrypted training.
y_true_dt = [int(v) for v in df_test_local["is_cancer"].values]

dt_enc_scores = []
for _, row in df_test_local.iterrows():
    _, risk = bc_dt_predict(dt, row.to_dict())
    dt_enc_scores.append(recalibrate_risk(risk, cohort_prev, POP_5YR_RATE))

plain_dt_proba = plaintext_predict_proba(
    plain_dt["model"], plain_dt["col_names"], df_test_local, feature_values, encoding="dt",
)
plain_dt_scores = [recalibrate_risk(prob, cohort_prev, POP_5YR_RATE) for prob in plain_dt_proba]

dt_m = compute_bc_metrics(
    y_true_dt, dt_enc_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)
pdt_m = compute_bc_metrics(
    y_true_dt, plain_dt_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)

print(f"Clinical threshold: {CLINICAL_THRESHOLD*100:.2f}% 5-year risk (FDA chemoprevention)")
print(
    f"Encrypted DT F1={dt_m['f1']:.3f} ROC-AUC={dt_m['roc_auc']:.3f} | "
    f"sklearn F1={pdt_m['f1']:.3f} ROC-AUC={pdt_m['roc_auc']:.3f}"
)
display(HTML(bc_model_summary_table(
    "Decision Tree",
    enc_metrics=dt_m,
    plain_metrics=pdt_m,
    enc_train_time=dt["train_time"],
    plain_train_time=plain_dt["train_time"],
    enc_queries=dt["total_aggregate_calls"],
    plain_label="sklearn CART",
)))
display(HTML(bc_confusion_matrix_html("Decision Tree", dt_m, pdt_m)))
print("\n-> Encrypted DT trained from BI aggregate counts only; no local training DataFrame was used.")


Training Decision Trees...
  Blind Insight Encrypted DT (aggregate counts only)... done (715.90s)
  Reused 68 NB marginals; additional DT aggregate queries: 162
  Plaintext DT (sklearn CART)... done (33ms)

[relatives_0=Yes]:
  [age_group_40_49=Yes]:
    [afb_30_plus=Yes]:
      -> risk=0.0222 (n=4364, pos=97)
    [afb_30_plus=No]:
      -> risk=0.0137 (n=16775, pos=230)
  [age_group_40_49=No]:
    [age_group_50_59=Yes]:
      -> risk=0.0251 (n=25427, pos=637)
    [age_group_50_59=No]:
      -> risk=0.0425 (n=38316, pos=1627)
[relatives_0=No]:
  [age_group_40_49=Yes]:
    [biopsies_0=Yes]:
      -> risk=0.0281 (n=3240, pos=91)
    [biopsies_0=No]:
      -> risk=0.0598 (n=569, pos=34)
  [age_group_40_49=No]:
    [relatives_1=Yes]:
      -> risk=0.0659 (n=9602, pos=633)
    [relatives_1=No]:
      -> risk=0.1136 (n=1707, pos=194)
Clinical threshold: 1.67% 5-year risk (FDA chemoprevention)
Encrypted DT F1=0.098 ROC-AUC=0.640 | sklearn F1=0.098 ROC-AUC=0.640


,sklearn CART,Blind Insight,Delta
F1 @1.67% risk,0.098,0.098,+0.000
F1@best,0.109,0.109,+0.000
ROC-AUC,0.640,0.640,+0.000
PR-AUC,0.063,0.063,+0.000
F1@best @ 1.6% pop prior,0.109,0.109,+0.000
Sensitivity,69.7%,69.7%,+0.0pp
Specificity,50.8%,50.8%,+0.0pp
PPV (precision),5.2%,5.2%,+0.0pp
Flagged High-Risk,50.0%,50.0%,+0.0pp
Accuracy,51.5%,51.5%,+0.0pp


,Pred Low,Pred High
Actual No,"4,891","4,733"
Actual Yes,114,262
,Pred Low,Pred High
Actual No,"4,891","4,733"
Actual Yes,114,262



-> Encrypted DT trained from BI aggregate counts only; no local training DataFrame was used.


### Encrypted Random Forest from BI Aggregate Counts

Random Forest trains an ensemble of count-backed trees through `RandomForestModel.fit_from_counts(...)`. It reuses the Decision Tree aggregate cache, asks BI only for missing split counts, and applies `HIPAA_K` to every tree split.


In [7]:
from blind_ml.healthcare import (
    run_encrypted_rf_bc, bc_rf_predict, bc_rf_describe,
    train_plaintext_rf_bc, bc_plaintext_rf_predict_proba,
)

print("Training Random Forest...")
print("  Blind Insight Encrypted RF (aggregate counts only)...", end=" ", flush=True)
rf = run_encrypted_rf_bc(
    client, ORG, DATASET, SCHEMA,
    dt_result=dt,
    n_estimators=7,
    max_depth=3,
    max_features=4,
    k_min=HIPAA_K,
    random_state=42,
)
print(f"done ({rf['train_time']:.2f}s)")
print(
    f"  Reused cache entries: {rf['reused_cache_entries']}; "
    f"additional RF aggregate queries: {rf['additional_rf_queries']}"
)

print("  Plaintext RF (sklearn)...", end=" ", flush=True)
plain_rf = train_plaintext_rf_bc(
    df_train, feature_values,
    n_estimators=7,
    max_depth=3,
    max_features=4,
    random_state=42,
)
print(f"done ({plain_rf['train_time'] * 1000:.0f}ms)")
print("\n" + bc_rf_describe(rf, max_trees=2))

# Evaluate on local held-out mirror; this is benchmark/evaluation only, not encrypted training.
y_true_rf = [int(v) for v in df_test_local["is_cancer"].values]

rf_enc_scores = []
for _, row in df_test_local.iterrows():
    _, risk = bc_rf_predict(rf, row.to_dict())
    rf_enc_scores.append(recalibrate_risk(risk, cohort_prev, POP_5YR_RATE))

rf_plain_proba = bc_plaintext_rf_predict_proba(
    plain_rf["model"], plain_rf["col_names"], df_test_local, feature_values,
)
rf_plain_scores = [recalibrate_risk(prob, cohort_prev, POP_5YR_RATE) for prob in rf_plain_proba]

rf_m = compute_bc_metrics(
    y_true_rf, rf_enc_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)
prf_m = compute_bc_metrics(
    y_true_rf, rf_plain_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)

print(
    f"Encrypted RF F1={rf_m['f1']:.3f} ROC-AUC={rf_m['roc_auc']:.3f} | "
    f"sklearn F1={prf_m['f1']:.3f} ROC-AUC={prf_m['roc_auc']:.3f}"
)
display(HTML(bc_model_summary_table(
    "Random Forest",
    enc_metrics=rf_m,
    plain_metrics=prf_m,
    enc_train_time=rf["train_time"],
    plain_train_time=plain_rf["train_time"],
    enc_queries=rf["total_aggregate_calls"],
    plain_label="sklearn RF",
)))
display(HTML(bc_confusion_matrix_html("Random Forest", rf_m, prf_m)))
print("\n-> Encrypted RF trained from BI aggregate counts only; no local training DataFrame was used.")


Training Random Forest...
  Blind Insight Encrypted RF (aggregate counts only)... done (1023.97s)
  Reused cache entries: 230; additional RF aggregate queries: 232
  Plaintext RF (sklearn)... done (49ms)

Random Forest: 7 trees, max_depth=3, max_features=4

Tree 1 features: age_group, race, relatives, biopsies, atypical, menarche, density, afb
[relatives_0=Yes]:
  [age_group_40_49=Yes]:
    [afb_30_plus=Yes]:
      -> risk=0.0222 (n=4364, pos=97)
    [afb_30_plus=No]:
      -> risk=0.0137 (n=16775, pos=230)
  [age_group_40_49=No]:
    [age_group_50_59=Yes]:
      -> risk=0.0251 (n=25427, pos=637)
    [age_group_50_59=No]:
      -> risk=0.0425 (n=38316, pos=1627)
[relatives_0=No]:
  [age_group_40_49=Yes]:
    [biopsies_0=Yes]:
      -> risk=0.0281 (n=3240, pos=91)
    [biopsies_0=No]:
      -> risk=0.0598 (n=569, pos=34)
  [age_group_40_49=No]:
    [relatives_1=Yes]:
      -> risk=0.0659 (n=9602, pos=633)
    [relatives_1=No]:
      -> risk=0.1136 (n=1707, pos=194)

Tree 2 features: bio

,sklearn RF,Blind Insight,Delta
F1 @1.67% risk,0.106,0.100,-0.006
F1@best,0.120,0.130,+0.010
ROC-AUC,0.660,0.668,+0.008
PR-AUC,0.069,0.080,+0.011
F1@best @ 1.6% pop prior,0.120,0.130,+0.010
Sensitivity,58.8%,75.3%,+16.5pp
Specificity,62.9%,47.8%,-15.1pp
PPV (precision),5.8%,5.3%,-0.5pp
Flagged High-Risk,38.0%,53.1%,+15.2pp
Accuracy,62.7%,48.8%,-13.9pp


,Pred Low,Pred High
Actual No,"4,596","5,028"
Actual Yes,93,283
,Pred Low,Pred High
Actual No,"6,049","3,575"
Actual Yes,155,221



-> Encrypted RF trained from BI aggregate counts only; no local training DataFrame was used.


### Encrypted AdaBoost from BI Aggregate Counts

AdaBoost trains count-backed decision stumps with `AdaBoostStumpModel.fit_from_counts(...)`. It reuses Random Forest cache first, then Decision Tree cache, and applies `HIPAA_K` so unsafe stump regions are skipped.


In [8]:
from blind_ml.healthcare import (
    run_encrypted_adaboost_bc, bc_adaboost_predict, bc_adaboost_describe,
    train_plaintext_adaboost_bc, bc_plaintext_adaboost_predict_proba,
)

print("Training AdaBoost...")
print("  Blind Insight Encrypted AdaBoost (aggregate counts only)...", end=" ", flush=True)
adaboost = run_encrypted_adaboost_bc(
    client, ORG, DATASET, SCHEMA,
    rf_result=rf,
    dt_result=dt,
    n_estimators=10,
    learning_rate=1.0,
    k_min=HIPAA_K,
)
print(f"done ({adaboost['train_time']:.2f}s)")
print(
    f"  Reused cache entries: {adaboost['reused_cache_entries']}; "
    f"additional AdaBoost aggregate queries: {adaboost['additional_adaboost_queries']}"
)

print("  Plaintext AdaBoost (sklearn)...", end=" ", flush=True)
plain_adaboost = train_plaintext_adaboost_bc(
    df_train, feature_values,
    n_estimators=10,
    learning_rate=1.0,
    random_state=42,
)
print(f"done ({plain_adaboost['train_time'] * 1000:.0f}ms)")
print("\n" + bc_adaboost_describe(adaboost, max_stumps=5))

# Evaluate on local held-out mirror; this is benchmark/evaluation only, not encrypted training.
y_true_adaboost = [int(v) for v in df_test_local["is_cancer"].values]

adaboost_enc_scores = []
for _, row in df_test_local.iterrows():
    _, risk = bc_adaboost_predict(adaboost, row.to_dict())
    adaboost_enc_scores.append(recalibrate_risk(risk, cohort_prev, POP_5YR_RATE))

adaboost_plain_proba = bc_plaintext_adaboost_predict_proba(
    plain_adaboost["model"], plain_adaboost["col_names"], df_test_local, feature_values,
)
adaboost_plain_scores = [recalibrate_risk(prob, cohort_prev, POP_5YR_RATE) for prob in adaboost_plain_proba]

adaboost_m = compute_bc_metrics(
    y_true_adaboost, adaboost_enc_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)
padaboost_m = compute_bc_metrics(
    y_true_adaboost, adaboost_plain_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)

print(
    f"Encrypted AdaBoost F1={adaboost_m['f1']:.3f} ROC-AUC={adaboost_m['roc_auc']:.3f} | "
    f"sklearn F1={padaboost_m['f1']:.3f} ROC-AUC={padaboost_m['roc_auc']:.3f}"
)
display(HTML(bc_model_summary_table(
    "AdaBoost",
    enc_metrics=adaboost_m,
    plain_metrics=padaboost_m,
    enc_train_time=adaboost["train_time"],
    plain_train_time=plain_adaboost["train_time"],
    enc_queries=adaboost["total_aggregate_calls"],
    plain_label="sklearn AdaBoost",
)))
display(HTML(bc_confusion_matrix_html("AdaBoost", adaboost_m, padaboost_m)))
print("\n-> Encrypted AdaBoost trained from BI aggregate counts only; no local training DataFrame was used.")


Training AdaBoost...
  Blind Insight Encrypted AdaBoost (aggregate counts only)... done (0.00s)
  Reused cache entries: 462; additional AdaBoost aggregate queries: 0
  Plaintext AdaBoost (sklearn)... done (146ms)

Empty AdaBoost model
Encrypted AdaBoost F1=0.000 ROC-AUC=0.500 | sklearn F1=0.072 ROC-AUC=0.651


,sklearn AdaBoost,Blind Insight,Delta
F1 @1.67% risk,0.072,0.000,-0.072
F1@best,0.122,0.072,-0.049
ROC-AUC,0.651,0.500,-0.151
PR-AUC,0.060,0.038,-0.022
F1@best @ 1.6% pop prior,0.122,0.072,-0.049
Sensitivity,100.0%,0.0%,-100.0pp
Specificity,0.0%,100.0%,+100.0pp
PPV (precision),3.8%,0.0%,-3.8pp
Flagged High-Risk,100.0%,0.0%,-100.0pp
Accuracy,3.8%,96.2%,+92.5pp


,Pred Low,Pred High
Actual No,"9,624",0
Actual Yes,376,0
,Pred Low,Pred High
Actual No,0,"9,624"
Actual Yes,0,376



-> Encrypted AdaBoost trained from BI aggregate counts only; no local training DataFrame was used.


 ### Encrypted Logistic Regression (Aggregate OLS/ridge + HIPAA k=11)

 Naive Bayes assumes features are independent. **Logistic Regression** captures feature
 interactions by learning from **BI pairwise cross-tabulations** and solving an aggregate
 OLS/ridge system.

 ```
 β = (X'X + λI)^-1 X'y
 X'y: BI class-split NB marginals
 X'X: BI pairwise aggregate counts
 λ:   ridge_lambda=0.01
 ```

 **Encrypted LR path:** aggregate OLS/ridge + HIPAA k=11, no local IRLS. Local DataFrames are
 used only for the sklearn plaintext benchmark and held-out evaluation.

 **HIPAA compliance via CMS cell suppression (k=11):**
 The [CMS Cell Suppression Policy](https://resdac.org/node/1506) prohibits reporting any
 aggregate cell with a count of 1-10 to prevent patient re-identification. Pairwise cells keep
 exact zeroes, preserve counts >= 11, and replace counts 1-10 with the independence estimate
 `(count_A * count_B / N)` before entering the model.


In [9]:
from blind_ml.healthcare import (
    run_encrypted_lr_bc_ols, bc_lr_predict,
    train_plaintext_bc_lr, plaintext_predict_proba,
    bc_lr_summary_table,
)

print("Training Logistic Regression models...")
print("  Encrypted LR: aggregate OLS/ridge + HIPAA k=11, no local IRLS...", end=" ", flush=True)
lr_model = run_encrypted_lr_bc_ols(
    client, ORG, DATASET, SCHEMA,
    feature_values=feature_values,
    bi_raw=bi_raw,
    ridge_lambda=0.01,
    min_cell_size=HIPAA_K,
)
print(f"done ({lr_model['train_time']:.1f}s)")
print(f"  BI pairwise queries: {lr_model['pairwise_queries']}")
print(f"  HIPAA suppression: {lr_model['n_suppressed']} pairwise cells replaced (CMS k={HIPAA_K})")

print("  Plaintext Logistic Regression (sklearn benchmark)...", end=" ", flush=True)
plain_lr = train_plaintext_bc_lr(df_train, feature_values)
print(f"done ({plain_lr['train_time'] * 1000:.0f}ms)")

lr_enc_scores = []
for _, row in df_test_local.iterrows():
    _, risk = bc_lr_predict(lr_model, row.to_dict(), use_sigmoid=True)
    lr_enc_scores.append(recalibrate_risk(risk, cohort_prev, POP_5YR_RATE))

plain_lr_proba = plaintext_predict_proba(plain_lr["model"], plain_lr["col_names"], df_test_local, feature_values)
lr_plain_scores = [recalibrate_risk(prob, cohort_prev, POP_5YR_RATE) for prob in plain_lr_proba]
y_true_lr = [int(v) for v in df_test_local["is_cancer"].values]

lr_m = compute_bc_metrics(
    y_true_lr, lr_enc_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)
plr_m = compute_bc_metrics(
    y_true_lr, lr_plain_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)

print(f"  Encrypted LR F1={lr_m['f1']:.3f} ROC-AUC={lr_m['roc_auc']:.3f} | sklearn F1={plr_m['f1']:.3f} ROC-AUC={plr_m['roc_auc']:.3f}")
print(f"  Suppression policy: {lr_model['suppression_policy']}")

print()
print(f"Clinical threshold: {CLINICAL_THRESHOLD*100:.2f}% 5-year risk (FDA chemoprevention)")
display(HTML(bc_lr_summary_table(
    enc_train_time=lr_model["train_time"], plain_train_time=plain_lr["train_time"],
    enc_queries=lr_model["enc_queries"], new_queries=lr_model["pairwise_queries"],
    enc_f1=lr_m["f1"], plain_f1=plr_m["f1"],
    enc_sens=lr_m["sens"], plain_sens=plr_m["sens"],
    enc_spec=lr_m["spec"], plain_spec=plr_m["spec"],
    enc_ppv=lr_m["ppv"], plain_ppv=plr_m["ppv"],
    enc_flagged=lr_m["flagged"], plain_flagged=plr_m["flagged"],
    n_suppressed=lr_model["n_suppressed"], cms_k=HIPAA_K,
)))
display(HTML(bc_confusion_matrix_html("Logistic Regression", lr_m, plr_m)))


Training Logistic Regression models...
  Encrypted LR: aggregate OLS/ridge + HIPAA k=11, no local IRLS... done (239.9s)
  BI pairwise queries: 502
  HIPAA suppression: 2 pairwise cells replaced (CMS k=11)
  Plaintext Logistic Regression (sklearn benchmark)... done (69ms)
  Encrypted LR F1=0.072 ROC-AUC=0.593 | sklearn F1=0.114 ROC-AUC=0.678
  Suppression policy: cms_k11_pairwise_independence_estimate

Clinical threshold: 1.67% 5-year risk (FDA chemoprevention)


,Plaintext,Blind Insight,Overhead
F1 Score,0.114,0.072,-4.2pp
Sensitivity (recall),59.0%,100.0%,+41.0pp
Specificity,65.8%,0.0%,-65.8pp
PPV (precision),6.3%,3.8%,-2.6pp
Flagged High-Risk,35.1%,100.0%,+64.9pp
BI Queries,502,502,-
New Queries,-,502,-
HIPAA Suppressed (k=11),-,2,-
Train Time,69ms,239.9s,+239.8s
Data Decrypted,YES,NEVER,-


,Pred Low,Pred High
Actual No,0,"9,624"
Actual Yes,0,376
,Pred Low,Pred High
Actual No,"6,337","3,287"
Actual Yes,154,222


 ### Encrypted Histogram Classifier from BI Marginal Counts

 Histogram Classifier uses the same class-split BI marginal counts as Naive Bayes, but each
 `feature=value` bucket votes directly with `P(cancer | feature=value)`. It does not multiply
 feature conditionals, and it never reads local rows in the encrypted training path.

 The encrypted path reuses the HIPAA k=11-suppressed NB marginals already in `bi_raw`; the
 plaintext benchmark trains the same histogram algorithm from the local mirror only.


In [10]:
from blind_ml.healthcare import (
    run_encrypted_histogram_bc, bc_histogram_predict,
    train_plaintext_histogram_bc, bc_plaintext_histogram_predict_proba,
)

print("Training Histogram Classifier...")
print("  Encrypted Histogram from BI marginals...", end=" ", flush=True)
hist_enc = run_encrypted_histogram_bc(
    client, ORG, DATASET, SCHEMA,
    feature_values=feature_values,
    bi_raw=bi_raw,
    min_cell_size=HIPAA_K,
)
print(f"done ({hist_enc['train_time']:.2f}s)")
print(f"  Reused source: {hist_enc['raw_results_source']}; new marginal queries: {hist_enc['marginal_queries']}")
print(f"  HIPAA/CMS k={hist_enc['min_cell_size']} suppressed cells: {hist_enc['n_suppressed']}")

print("  Plaintext Histogram benchmark...", end=" ", flush=True)
hist_plain = train_plaintext_histogram_bc(df_train, feature_values)
print(f"done ({hist_plain['train_time'] * 1000:.0f}ms)")

hist_enc_scores = []
for _, row in df_test_local.iterrows():
    _, risk = bc_histogram_predict(hist_enc, row.to_dict())
    hist_enc_scores.append(recalibrate_risk(risk, cohort_prev, POP_5YR_RATE))

hist_plain_proba = bc_plaintext_histogram_predict_proba(hist_plain, df_test_local)
hist_plain_scores = [recalibrate_risk(prob, cohort_prev, POP_5YR_RATE) for prob in hist_plain_proba]
y_true_hist = [int(v) for v in df_test_local["is_cancer"].values]

hist_m = compute_bc_metrics(
    y_true_hist, hist_enc_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)
hist_plain_m = compute_bc_metrics(
    y_true_hist, hist_plain_scores,
    threshold=CLINICAL_THRESHOLD,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
)

print(
    f"Encrypted Histogram F1={hist_m['f1']:.3f} ROC-AUC={hist_m['roc_auc']:.3f} | "
    f"plaintext Histogram F1={hist_plain_m['f1']:.3f} ROC-AUC={hist_plain_m['roc_auc']:.3f}"
)
print(f"Suppression policy: {hist_enc['suppression_policy']}")
display(HTML(bc_model_summary_table(
    "Histogram Classifier",
    enc_metrics=hist_m,
    plain_metrics=hist_plain_m,
    enc_train_time=hist_enc["train_time"],
    plain_train_time=hist_plain["train_time"],
    enc_queries=hist_enc["enc_queries"],
    plain_label="Plaintext Histogram",
)))
display(HTML(bc_confusion_matrix_html("Histogram Classifier", hist_m, hist_plain_m)))


Training Histogram Classifier...
  Encrypted Histogram from BI marginals... done (0.00s)
  Reused source: bi_raw; new marginal queries: 0
  HIPAA/CMS k=11 suppressed cells: 0
  Plaintext Histogram benchmark... done (168ms)
Encrypted Histogram F1=0.121 ROC-AUC=0.676 | plaintext Histogram F1=0.121 ROC-AUC=0.676
Suppression policy: cms_k11_fixed_midpoint_5


,Plaintext Histogram,Blind Insight,Delta
F1 @1.67% risk,0.121,0.121,+0.000
F1@best,0.136,0.136,+0.000
ROC-AUC,0.676,0.676,-0.000
PR-AUC,0.078,0.078,+0.000
F1@best @ 1.6% pop prior,0.136,0.136,+0.000
Sensitivity,48.9%,48.9%,+0.0pp
Specificity,74.2%,74.2%,+0.0pp
PPV (precision),6.9%,6.9%,+0.0pp
Flagged High-Risk,26.7%,26.7%,+0.0pp
Accuracy,73.2%,73.2%,+0.0pp


,Pred Low,Pred High
Actual No,"7,138","2,486"
Actual Yes,192,184
,Pred Low,Pred High
Actual No,"7,138","2,486"
Actual Yes,192,184


 ### Eight-Model Encrypted ML Comparison

 This table compares all encrypted ML models trained above. Clinical BCRAT/BCSC comparison
 remains separate because those are published risk calculators rather than encrypted ML models.


In [11]:
eight_model_rows = [
    {"name": "Naive Bayes", "enc_metrics": nb_m},
    {"name": "Gaussian NB", "enc_metrics": gnb_enc_m},
    {"name": "Bayesian Network", "enc_metrics": bn_enc_m},
    {"name": "Decision Tree", "enc_metrics": dt_m},
    {"name": "Random Forest", "enc_metrics": rf_m},
    {"name": "AdaBoost", "enc_metrics": adaboost_m},
    {"name": "Logistic Regression", "enc_metrics": lr_m},
    {"name": "Histogram", "enc_metrics": hist_m},
]

display(HTML(bc_eight_model_table(eight_model_rows)))
print("All eight encrypted ML models trained from BI aggregate counts; local rows were used only for plaintext benchmarks and held-out evaluation.")


Metric,Naive Bayes,Gaussian NB,Bayesian Network,Decision Tree,Random Forest,AdaBoost,Logistic Regression,Histogram
F1 @1.67%,0.112,0.103,0.107,0.098,0.100,0.000,0.072,0.121
F1@best,0.126,0.129,0.136,0.109,0.130,0.072,0.116,0.136
ROC-AUC,0.675,0.648,0.678,0.640,0.668,0.500,0.593,0.676
PR-AUC,0.074,0.066,0.078,0.063,0.080,0.038,0.062,0.078
Sensitivity,58.5%,67.6%,63.3%,69.7%,75.3%,0.0%,100.0%,48.9%
Specificity,65.5%,55.1%,60.3%,50.8%,47.8%,100.0%,0.0%,74.2%
PPV,6.2%,5.5%,5.9%,5.2%,5.3%,0.0%,3.8%,6.9%
Flagged High-Risk,35.4%,45.8%,40.6%,50.0%,53.1%,0.0%,100.0%,26.7%


All eight encrypted ML models trained from BI aggregate counts; local rows were used only for plaintext benchmarks and held-out evaluation.


 ### Real-Time Scoring: Encrypted Records

 Fetch encrypted patient records from Blind Insight, decrypt locally for display (Data Owner role),
 and score the privacy-preserving NB model alongside the published clinical calculators.

 The encrypted ML models above were trained **entirely on BI aggregates** -- none saw individual
 training records. BCRAT and BCSC use published clinical formulas.


In [12]:
print("Fetching encrypted + decrypted records from BI...", end=" ", flush=True)
rt = run_bc_realtime_demo(
    client, ORG, DATASET, SCHEMA,
    P_cancer, P_no_cancer, P_enc,
    sample_size=50,
)
print("done")

print(
    f"Query: {rt['enc_query_time'] * 1000:.0f}ms for {rt['rt_count']} records | "
    f"Total: {rt['total_query_time'] * 1000:.0f}ms (encrypted + decrypted)"
)
display(HTML(rt["html"]))

Fetching encrypted + decrypted records from BI... done
Query: 2897ms for 50 records | Total: 5160ms (encrypted + decrypted)


age,num_first_degree_relatives,breast_density,menarche_category,atypical_hyperplasia,BI Risk,BCRAT,BCSC
e698a79e860805ee51b57a2fc52446bc7d7,a2a008c6e350e5132640603c028f43196aa,5dc38aa5794f05def0b5a72adffaa5aff3b,170aba2c3194c12c9ed9dc5cb2bf16e211b,397e9643dedb1a7183e4b6fe0f7853ee706,0.66%,0.77%,0.80%
d1fbdc711068f589f057d74562c8c5c4b0b,03ef927cb549fa2e1f712f1223ae2830b8c,15c47addb6774c9752717864999aa1447ae,4dc39533e2495250fd1c8a6748f502a97a8,7868de0abbf4860ed3eca3c51a9584b2ba6,0.39%,0.44%,0.48%
b3d568c2869f4d384769935198b73911add,e1c8f9b2a00c2d65ee77a9ef7645547fe67,2f48cd05353b90565ed58511dc0bd606bf2,1186ef65a98257609ec8e98a622649a165d,97440621edcd3435afcf9fd1b550b99f81e,1.41%,0.93%,1.26%
be9ff6fb0330c40bfde62427edfae8f98ee,82888f30645e629e37362a0f8182f12259f,6ba11de78f2c09632f699308d749c4356f7,68703d0376d360f973d17581b691d92c12c,5616cfff6305760a453a24ec2e6df246739,0.47%,0.50%,0.48%
a44d28666d9cd1589ac556392dc879df179,178990b6c2ab780e9d2f11f77d0c495df68,80ff1eb6865b1f54abc057bd7c85481c637,e1a7eff3f957191b25afe96895db9249f6d,5a64343d33ebc3372cef79442afba924aba,0.85%,1.10%,1.14%


 ### Spot-Check Validation: NB, Decision Tree, and Aggregate LR

 This legacy validation cell compares three representative encrypted models against sklearn
 plaintext benchmarks. The final ML comparison above is the authoritative eight-model summary.

 Logistic Regression here uses the aggregate OLS/ridge model trained from BI pairwise counts;
 there is no local IRLS in the encrypted path.


In [13]:
from blind_ml.healthcare import run_bc_full_validation

df_test, _ = load_bc_test_data(TEST_SQLITE_DB)
print(f"  Test set: {len(df_test):,} records")

print("  Validating NB, DT, and aggregate LR (BI vs plaintext sklearn)...", end=" ", flush=True)
tv = run_bc_full_validation(
    df_test,
    P_cancer, P_no_cancer, P_enc,
    P_cancer_plain, P_no_cancer_plain, P_plain,
    dt_model=dt,
    plain_dt_model=plain_dt["model"],
    plain_dt_col_names=plain_dt["col_names"],
    lr_beta=lr_model["beta"], lr_dummy_index=lr_model["dummy_index"],
    plain_lr_model=plain_lr["model"],
    plain_lr_col_names=plain_lr["col_names"],
    feature_values=feature_values,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
    threshold=CLINICAL_THRESHOLD,
    use_sigmoid=True,
)
print(f"done ({tv['pred_time']:.1f}s)")
print(f"  NB agreement: {tv['nb_agreement']*100:.1f}% | DT agreement: {tv['dt_agreement']*100:.1f}% | LR agreement: {tv['lr_agreement']*100:.1f}%")
display(HTML(tv["metrics_html"]))
display(HTML(tv["cm_html"]))


  Test set: 10,000 records
  Validating NB, DT, and aggregate LR (BI vs plaintext sklearn)... done (0.1s)
  NB agreement: 100.0% | DT agreement: 100.0% | LR agreement: 35.1%


,Naive Bayes,Decision Tree,Logistic Reg
Test records,"10,000","10,000","10,000"
Cancer (5yr),376,376,376
No Cancer,"9,624","9,624","9,624"
BI ↔ Plaintext Agreement,100.0%,100.0%,35.1%
F1 (encrypted),0.112,0.098,0.072
F1 (plaintext),0.112,0.098,0.114
F1 Δ (enc − plain),0pp,0pp,-4.2pp
Sensitivity (encrypted),58.5%,69.7%,100.0%
Specificity (encrypted),65.5%,50.8%,0.0%
PPV (encrypted),6.2%,5.2%,3.8%


,Pred Low,Pred High
Actual No,"6,303","3,321"
Actual Yes,156,220
,Pred Low,Pred High
Actual No,"6,303","3,321"
Actual Yes,156,220
,Pred Low,Pred High
Actual No,"4,891","4,733"
Actual Yes,114,262
,Pred Low,Pred High
Actual No,"4,891","4,733"


 ### Model Comparison: BI Naive Bayes vs BCRAT vs BCSC

 Now compare the privacy-preserving NB model against two published clinical
 risk calculators at standard clinical thresholds:

 | Threshold | Meaning |
 |-----------|----------|
 | **1.67%** | FDA threshold for chemoprevention eligibility |
 | **3.0%**  | Enhanced screening recommendation (MRI) |

 **Paige et al. (2023)** showed ~20% disagreement between BCRAT, BCSC, and IBIS
 at the individual patient level, even when population-level performance is similar.
 This motivates risk-based screening with multiple models.

In [14]:
print("  Scoring all test records with NB, BCRAT, and BCSC...", end=" ", flush=True)
comp_start = time.time()
comp = run_model_comparison(df_test, P_cancer, P_no_cancer, P_enc)
comp_time = time.time() - comp_start
print(f"done ({comp_time:.1f}s)")

display(HTML(comp["summary_html"]))
display(HTML(sample_comparison_table(comp["comp_df"], limit=15)))

print(f"""\n--- Key Takeaways ---
1. BCRAT flags {comp['pct_bcrat_167']:.0f}% as high-risk at 1.67% (Paige study: 35.5%)
2. BCSC flags {comp['pct_bcsc_167']:.0f}% as high-risk at 1.67% (Paige study: 20.0%)
3. Models agree on {comp['all_agree']*100:.0f}% of patients (Paige study: ~79%)
4. The BI Naive Bayes model participated in this comparison
   WITHOUT any institution sharing raw patient data.
""")

  Scoring all test records with NB, BCRAT, and BCSC... done (0.2s)


,BI Naive Bayes,BCRAT (Gail),BCSC
% Flagged High Risk (>=1.67%),35.4%,37.7%,28.8%
% Flagged High Risk (>=3.0%),9.0%,9.1%,7.6%
Sensitivity (cancer cases at 1.67%),58.5%,58.2%,48.9%
NB-BCRAT Agreement,85.9%,-,-
NB-BCSC Agreement,83.8%,-,-
BCRAT-BCSC Agreement,-,73.3%,-
All 3 Models Agree,71.5%,-,-


Age,BI Risk,BCRAT Risk,BCSC Risk,Actual (5yr),All Agree
45,1.41%,1.58%,0.91%,❌ Cancer,✓
49,0.83%,1.23%,0.56%,✅ No,✓
49,0.81%,0.75%,0.61%,✅ No,✓
50,1.77%,1.19%,2.09%,✅ No,✗
50,0.66%,0.77%,0.80%,✅ No,✓
54,1.01%,0.87%,1.35%,❌ Cancer,✓
57,0.87%,0.77%,1.35%,✅ No,✓
63,1.61%,1.36%,1.72%,✅ No,✗
70,2.47%,2.60%,1.92%,✅ No,✓
71,1.58%,1.54%,2.56%,✅ No,✗



--- Key Takeaways ---
1. BCRAT flags 38% as high-risk at 1.67% (Paige study: 35.5%)
2. BCSC flags 29% as high-risk at 1.67% (Paige study: 20.0%)
3. Models agree on 72% of patients (Paige study: ~79%)
4. The BI Naive Bayes model participated in this comparison
   WITHOUT any institution sharing raw patient data.

